# Seattle Crime & Housing Price Analysis

This notebook integrates Seattle Police Department (SPD) crime incident data with KC Assessor housing sales to quantify the **crime discount** embedded in residential property prices.

> **Geographic scope:** SPD data covers **Seattle city limits only**, so the housing analysis in this notebook is restricted to Seattle School District properties.

**Data sources**
| Source | Coverage | API |
|--------|----------|-----|
| [SPD Crime Data 2008–Present](https://data.seattle.gov/Public-Safety/SPD-Crime-Data-2008-Present/tazs-3rd5) | Seattle city limits | Socrata (`tazs-3rd5`) |
| KC Assessor (existing) | King County | Already downloaded |
| KC Parcel coords (existing) | King County | `education_data/kc_parcel_coords.csv` |

**Sections**
1. Download & QC — SPD crime incidents 2015–2024
2. Crime trends — volume, types, seasonality
3. Spatial distribution — neighborhood crime map
4. Spatio-temporal decay feature — crime density per property at time of sale
5. Crime vs. housing price — the crime discount

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import geopandas as gpd
import folium
from folium.plugins import HeatMap
import requests
import warnings
from pathlib import Path

warnings.filterwarnings('ignore')
pd.set_option('display.float_format', '{:,.2f}'.format)
plt.rcParams['figure.dpi'] = 120
sns.set_theme(style='whitegrid')

fmt_dol = mticker.FuncFormatter(lambda x, _: f'${x:,.0f}')
fmt_k   = mticker.FuncFormatter(lambda x, _: f'{x/1_000:.0f}K')
ENC      = 'latin-1'
DATA_DIR  = Path('kc_assessor_data')
EDU_DIR   = Path('education_data')
CRIME_DIR = Path('crime_data')
CRIME_DIR.mkdir(exist_ok=True)

---
## 1. Download & QC

**Dataset used:** SPD Crime Data 2008–Present (`data.seattle.gov`, dataset ID `tazs-3rd5`)

**Goal:** Download Seattle crime incidents for 2015–2024 with valid GPS coordinates. Filter out records with redacted locations, understand the offense taxonomy, and check data quality.

In [ ]:
# ── Download SPD crime data 2015–2024 (valid coords only) ─────────────────────
# ~70K records/year × 10 years ≈ 700K rows; max 1,000/request → paginated.
# ~85% of records have non-redacted coordinates.

import time

CRIME_FILE = CRIME_DIR / 'spd_crime_2015_2024.csv'
BASE_URL   = 'https://data.seattle.gov/resource/tazs-3rd5.json'

if CRIME_FILE.exists():
    crime = pd.read_csv(CRIME_FILE, low_memory=False)
    print(f'Loaded cached file: {len(crime):,} records')
else:
    print('Downloading SPD crime data 2015–2024 (~700K records, ~15 min)...')
    all_records = []
    batch_size  = 1000
    offset      = 0
    where_clause = (
        "offense_date >= '2015-01-01' "
        "AND offense_date < '2025-01-01' "
        "AND latitude != 'REDACTED'"
    )
    start = time.time()

    while True:
        r = requests.get(BASE_URL, params={
            '$where':            where_clause,
            '$select':           'offense_id,offense_date,offense_category,'
                                 'offense_sub_category,nibrs_offense_code_description,'
                                 'nibrs_crime_against_category,latitude,longitude,neighborhood,beat',
            '$limit':            batch_size,
            '$offset':           offset,
            '$order':            'offense_date ASC',
        }, timeout=30)
        r.raise_for_status()
        batch = r.json()
        if not batch:
            break
        all_records.extend(batch)
        offset += len(batch)
        if offset % 100_000 == 0:
            elapsed = time.time() - start
            rate    = offset / elapsed
            eta     = (700_000 - offset) / rate
            print(f'  {offset:,} downloaded  ({eta/60:.1f} min remaining)')
        if len(batch) < batch_size:
            break

    crime = pd.DataFrame(all_records)
    crime.to_csv(CRIME_FILE, index=False)
    print(f'Done. {len(crime):,} records → {CRIME_FILE}')

# Parse types
crime['offense_date'] = pd.to_datetime(crime['offense_date'], errors='coerce')
crime['year']  = crime['offense_date'].dt.year
crime['month'] = crime['offense_date'].dt.month
crime['lat']   = pd.to_numeric(crime['latitude'],  errors='coerce')
crime['lon']   = pd.to_numeric(crime['longitude'], errors='coerce')
crime = crime.dropna(subset=['lat','lon','offense_date'])

print(f'\nClean records: {len(crime):,}')
print(f'Date range: {crime["offense_date"].min().date()} – {crime["offense_date"].max().date()}')
print(f'\noffense_category breakdown:')
print(crime['offense_category'].value_counts())

In [ ]:
# ── Define serious crimes ─────────────────────────────────────────────────────
# For the spatio-temporal decay feature we focus on crimes most likely to
# affect perceived safety and housing prices.

SERIOUS_CATEGORIES = {'VIOLENT CRIME', 'PROPERTY CRIME'}

SERIOUS_SUBTYPES = {
    'Burglary/Breaking & Entering',
    'Robbery',
    'Aggravated Assault',
    'Simple Assault',
    'Motor Vehicle Theft',
    'Arson',
    'Homicide Offenses',
    'Rape',
    'Kidnapping/Abduction',
}

crime['is_serious'] = (
    crime['offense_category'].isin(SERIOUS_CATEGORIES) |
    crime['nibrs_offense_code_description'].isin(SERIOUS_SUBTYPES)
)
crime['is_violent'] = crime['offense_category'] == 'VIOLENT CRIME'

print(f'Total incidents:     {len(crime):>10,}')
print(f'Serious crimes:      {crime["is_serious"].sum():>10,}  ({crime["is_serious"].mean()*100:.1f}%)')
print(f'Violent crimes:      {crime["is_violent"].sum():>10,}  ({crime["is_violent"].mean()*100:.1f}%)')
print(f'\nTop 15 offense types (serious only):')
print(
    crime[crime['is_serious']]['nibrs_offense_code_description']
    .value_counts().head(15).to_string()
)

---
## 2. Crime Trends

**Dataset used:** SPD crime incidents 2015–2024

**Goal:** Understand how overall crime volume and serious crime have changed over time in Seattle, and identify any seasonality patterns that might affect housing market perception.

In [ ]:
# ── Annual trends ─────────────────────────────────────────────────────────────
annual = crime.groupby('year').agg(
    total    = ('offense_id', 'count'),
    serious  = ('is_serious', 'sum'),
    violent  = ('is_violent', 'sum'),
)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(annual.index, annual['total'],   marker='o', label='All incidents',    color='steelblue')
axes[0].plot(annual.index, annual['serious'], marker='s', label='Serious crimes',   color='coral')
axes[0].plot(annual.index, annual['violent'], marker='^', label='Violent crimes',   color='crimson')
axes[0].yaxis.set_major_formatter(fmt_k)
axes[0].set_title('Seattle Crime Volume by Year (2015–2024)')
axes[0].set_xlabel('Year')
axes[0].set_ylabel('Incident Count')
axes[0].legend()

# Category mix over time
cat_yr = crime.groupby(['year','offense_category']).size().unstack(fill_value=0)
cat_yr.plot(kind='bar', stacked=True, ax=axes[1],
            color=['#e74c3c','#3498db','#95a5a6'], alpha=0.85)
axes[1].yaxis.set_major_formatter(fmt_k)
axes[1].set_title('Crime Category Mix by Year')
axes[1].set_xlabel('Year')
axes[1].tick_params(axis='x', rotation=45)
axes[1].legend(title='Category')

plt.tight_layout()
plt.show()
print(annual)

In [ ]:
# ── Seasonality ───────────────────────────────────────────────────────────────
month_stats = crime.groupby('month').agg(
    total   = ('offense_id', 'count'),
    serious = ('is_serious', 'sum'),
).div(crime['year'].nunique())   # average per year

mnames = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']

fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(mnames, month_stats['total'],   label='All incidents', color='steelblue', alpha=0.6)
ax.bar(mnames, month_stats['serious'], label='Serious crimes', color='coral',    alpha=0.8)
ax.yaxis.set_major_formatter(fmt_k)
ax.set_title('Average Monthly Crime Volume (2015–2024)')
ax.set_ylabel('Avg incidents / year')
ax.legend()
plt.tight_layout()
plt.show()

**Conclusions — Trends:**
- *(To be filled after execution — observe peak/trough years, COVID effect, recent trajectory)*
- *(Note whether violent and property crime trends diverge)*
- *(Seasonality: summer months typically see higher crime volume)*

---
## 3. Spatial Distribution

**Dataset used:** SPD crime incidents (lat/lon) · Seattle neighborhood boundaries

**Goal:** Identify which Seattle neighborhoods have the highest crime density, using both a heat map and a neighborhood-level bar chart.

In [ ]:
# ── Heat map: serious crime density across Seattle (2020–2024) ────────────────
recent_serious = crime[
    (crime['is_serious']) &
    (crime['year'].between(2020, 2024))
][['lat','lon']].dropna()

m_heat = folium.Map(location=[47.61, -122.33], zoom_start=12, tiles='CartoDB dark_matter')

HeatMap(
    data      = recent_serious[['lat','lon']].values.tolist(),
    radius    = 12,
    blur      = 15,
    min_opacity = 0.3,
    gradient  = {0.2: 'blue', 0.5: 'lime', 0.8: 'orange', 1.0: 'red'}
).add_to(m_heat)

folium.LayerControl().add_to(m_heat)
m_heat.save(str(CRIME_DIR / 'map_crime_heatmap.html'))
print(f'Heat map saved: crime_data/map_crime_heatmap.html  ({len(recent_serious):,} points)')
m_heat

In [ ]:
# ── Neighborhood ranking ──────────────────────────────────────────────────────
nbhd = (
    crime[crime['is_serious'] & crime['year'].between(2020,2024)]
    .groupby('neighborhood')
    .agg(total=('offense_id','count'), violent=('is_violent','sum'))
    .sort_values('total', ascending=False)
)
nbhd['pct_violent'] = nbhd['violent'] / nbhd['total'] * 100

top20 = nbhd.head(20)
bot20 = nbhd.tail(20).sort_values('total')

fig, axes = plt.subplots(1, 2, figsize=(16, 8))

for ax, data, title, color in [
    (axes[0], top20, 'Top 20 Highest-Crime Neighborhoods (2020–24)', 'crimson'),
    (axes[1], bot20, 'Top 20 Lowest-Crime Neighborhoods (2020–24)',  'steelblue'),
]:
    bars = ax.barh(data.index, data['total'], color=color, alpha=0.8)
    for bar, (_, row) in zip(bars, data.iterrows()):
        ax.text(bar.get_width()*1.01, bar.get_y()+bar.get_height()/2,
                f"{int(row['total']):,}  ({row['pct_violent']:.0f}% violent)",
                va='center', fontsize=8)
    ax.set_title(title, fontsize=11)
    ax.set_xlabel('Serious crime incidents')
    ax.set_xlim(right=data['total'].max()*1.55)

plt.tight_layout()
plt.show()

**Conclusions — Spatial:**
- *(To be filled after execution — note highest-crime neighborhoods and their proximity to downtown)*
- *(Note percent violent crime — neighborhoods with high violent crime % likely carry larger price discounts than pure property crime areas)*

---
## 4. Spatio-Temporal Decay Feature

**Dataset used:** SPD crime incidents · KC Assessor SFR sales (Seattle) · Parcel lat/lon

**Goal:** Build a dynamic crime exposure score for each housing sale: count serious crimes within **500 meters** in the **12 months before** the sale date. This is more informative than a static neighborhood crime rate because it captures *what the buyer actually perceived at the time of purchase*.

**Method:** sklearn `BallTree` with haversine metric for fast radius queries (much faster than GeoPandas buffer for large datasets).

In [ ]:
# ── Load Seattle SFR sales with coordinates ───────────────────────────────────
rp  = pd.read_csv(DATA_DIR/'RealPropertySales/EXTR_RPSale.csv',   low_memory=False, encoding=ENC)
rb  = pd.read_csv(DATA_DIR/'ResidentialBuilding/EXTR_ResBldg.csv', low_memory=False, encoding=ENC)

def make_pin(df):
    return df['Major'].astype(str).str.zfill(6) + df['Minor'].astype(str).str.zfill(4)

rp['PIN'] = make_pin(rp)
rb['PIN'] = make_pin(rb)
rp['DocumentDate'] = pd.to_datetime(rp['DocumentDate'], errors='coerce')
rp['SaleYear']     = rp['DocumentDate'].dt.year

# Arms-length SFR filter 2016–2024 (need 1 year lookback so start at 2016)
al = rp[
    (rp['SaleReason'] == 1) & (rp['SalePrice'] > 10_000) &
    (rp['PropertyClass'] == 8) & (rp['SaleYear'].between(2016, 2024))
].copy()
al_latest = al.sort_values('DocumentDate').drop_duplicates('PIN', keep='last')

rb_sfr = rb[
    (rb['NbrLivingUnits']==1) & (rb['SqFtTotLiving'].between(200,15_000)) &
    (rb['YrBuilt'].between(1870,2024))
].sort_values('BldgNbr').drop_duplicates('PIN', keep='first')

sales = al_latest.merge(
    rb_sfr[['PIN','SqFtTotLiving','BldgGrade','Condition','YrBuilt','Bedrooms','BathFullCount']],
    on='PIN', how='inner'
)

# Attach parcel coordinates
parcel_coords = pd.read_csv(EDU_DIR/'kc_parcel_coords.csv')[['PIN','LAT','LON']]
sales = sales.merge(parcel_coords, on='PIN', how='inner')
sales = sales.dropna(subset=['LAT','LON'])

# Filter to Seattle bounding box (approx)
SEATTLE_BBOX = dict(lat_min=47.49, lat_max=47.74, lon_min=-122.46, lon_max=-122.22)
seattle_sales = sales[
    sales['LAT'].between(SEATTLE_BBOX['lat_min'], SEATTLE_BBOX['lat_max']) &
    sales['LON'].between(SEATTLE_BBOX['lon_min'], SEATTLE_BBOX['lon_max'])
].copy()

print(f'King County SFR sales (2016–2024): {len(sales):,}')
print(f'Seattle SFR sales (bounding box):  {len(seattle_sales):,}')

In [ ]:
# ── Compute spatio-temporal crime score ───────────────────────────────────────
# For each sale: count serious crimes within 500m in 12 months before sale date.
# Uses BallTree haversine for O(n log n) spatial queries.

from sklearn.neighbors import BallTree

SCORE_FILE = CRIME_DIR / 'seattle_sales_crime_score.csv'

if SCORE_FILE.exists():
    scores_df = pd.read_csv(SCORE_FILE)
    seattle_sales = seattle_sales.merge(scores_df, on='PIN', how='left')
    print(f'Loaded cached crime scores: {len(scores_df):,} records')
else:
    print('Computing crime scores (this may take a few minutes)...')
    RADIUS_M   = 500
    LOOKBACK_D = 365
    R_EARTH    = 6_371_000  # meters
    RADIUS_RAD = RADIUS_M / R_EARTH  # radians for BallTree

    serious_crime = crime[crime['is_serious']].copy()
    serious_crime['date_int'] = serious_crime['offense_date'].astype(np.int64) // 10**9  # unix seconds

    # Build BallTree on crime locations (radians)
    crime_coords_rad = np.radians(serious_crime[['lat','lon']].values)
    tree = BallTree(crime_coords_rad, metric='haversine')

    crime_dates = serious_crime['offense_date'].values
    crime_violent = serious_crime['is_violent'].values

    sale_scores = []
    sale_coords_rad = np.radians(seattle_sales[['LAT','LON']].values)
    sale_dates      = seattle_sales['DocumentDate'].values

    for i, (coord_rad, sale_date) in enumerate(zip(sale_coords_rad, sale_dates)):
        # Find all crimes within 500m radius
        indices = tree.query_radius([coord_rad], r=RADIUS_RAD)[0]
        if len(indices) == 0:
            sale_scores.append({'crime_count_500m': 0, 'violent_count_500m': 0})
            continue
        # Filter to 12 months before sale
        nearby_dates   = crime_dates[indices]
        days_before    = (sale_date - nearby_dates).astype('timedelta64[D]').astype(float)
        in_window      = (days_before >= 0) & (days_before <= LOOKBACK_D)
        sale_scores.append({
            'crime_count_500m':   int(in_window.sum()),
            'violent_count_500m': int(crime_violent[indices][in_window].sum()),
        })
        if (i+1) % 5_000 == 0:
            print(f'  {i+1:,} / {len(seattle_sales):,} processed')

    scores_df = pd.DataFrame(sale_scores)
    scores_df['PIN'] = seattle_sales['PIN'].values
    scores_df.to_csv(SCORE_FILE, index=False)
    seattle_sales = seattle_sales.merge(scores_df, on='PIN', how='left')
    print(f'Done. Scores saved to {SCORE_FILE}')

print(f'\nCrime score summary:')
print(seattle_sales[['crime_count_500m','violent_count_500m']].describe().round(1))

---
## 5. Crime vs. Housing Price — The Crime Discount

**Dataset used:** Seattle SFR sales (2016–2024) with spatio-temporal crime scores

**Goal:** Quantify how much each additional nearby crime incident is associated with lower sale prices, controlling roughly for property size. Estimate the "crime discount" across crime exposure quintiles.

In [ ]:
# ── Scatter: crime count vs. sale price ───────────────────────────────────────
df = seattle_sales.dropna(subset=['crime_count_500m','SalePrice'])
df = df[df['SalePrice'].between(100_000, 5_000_000)]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Raw scatter (sample)
sample = df.sample(min(10_000, len(df)), random_state=42)
axes[0].scatter(sample['crime_count_500m'], sample['SalePrice'],
                alpha=0.05, s=5, color='steelblue', rasterized=True)

# Bin means
bins = pd.cut(sample['crime_count_500m'], bins=20)
bin_med = sample.groupby(bins, observed=True)['SalePrice'].median()
bin_x   = [b.mid for b in bin_med.index]
axes[0].plot(bin_x, bin_med.values, 'r-o', markersize=5, lw=2, label='Bin median')
axes[0].set_xlabel('Serious crimes within 500m (12 months before sale)')
axes[0].set_ylabel('Sale Price')
axes[0].yaxis.set_major_formatter(fmt_dol)
axes[0].set_title('Sale Price vs. Nearby Crime Count')
axes[0].legend()

# Violent crime scatter
axes[1].scatter(sample['violent_count_500m'], sample['SalePrice'],
                alpha=0.05, s=5, color='crimson', rasterized=True)
bins_v = pd.cut(sample['violent_count_500m'], bins=15)
bin_med_v = sample.groupby(bins_v, observed=True)['SalePrice'].median()
bin_xv    = [b.mid for b in bin_med_v.index]
axes[1].plot(bin_xv, bin_med_v.values, 'k-o', markersize=5, lw=2, label='Bin median')
axes[1].set_xlabel('Violent crimes within 500m (12 months before sale)')
axes[1].set_ylabel('Sale Price')
axes[1].yaxis.set_major_formatter(fmt_dol)
axes[1].set_title('Sale Price vs. Nearby Violent Crime Count')
axes[1].legend()

plt.tight_layout()
plt.show()

r_all     = df['crime_count_500m'].corr(df['SalePrice'])
r_violent = df['violent_count_500m'].corr(df['SalePrice'])
print(f'Correlation (all serious crime vs price):  r = {r_all:.3f}')
print(f'Correlation (violent crime vs price):      r = {r_violent:.3f}')

In [ ]:
# ── Crime quintile analysis ───────────────────────────────────────────────────
df['crime_quintile'] = pd.qcut(
    df['crime_count_500m'], q=5,
    labels=['Q1\nLowest\ncrime','Q2','Q3','Q4','Q5\nHighest\ncrime']
)

q_stats = df.groupby('crime_quintile', observed=True).agg(
    median_price    = ('SalePrice',      'median'),
    median_sqft     = ('SqFtTotLiving',  'median'),
    median_crime    = ('crime_count_500m','median'),
    n               = ('SalePrice',      'count'),
)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

colors = ['#27ae60','#2ecc71','#f39c12','#e67e22','#e74c3c']

bars = axes[0].bar(q_stats.index, q_stats['median_price'], color=colors)
for bar, (_, row) in zip(bars, q_stats.iterrows()):
    axes[0].text(bar.get_x()+bar.get_width()/2, bar.get_height()*1.01,
                 f'${row["median_price"]/1e3:.0f}K\n({int(row["median_crime"])} crimes)',
                 ha='center', fontsize=9, fontweight='bold')
axes[0].yaxis.set_major_formatter(fmt_dol)
axes[0].set_title('Median Sale Price by Crime Exposure Quintile\n(Seattle SFR, 2016–2024)', fontsize=11)
axes[0].set_ylabel('Median Sale Price')

# Price per sqft (controls for size)
df['ppsf'] = df['SalePrice'] / df['SqFtTotLiving'].replace(0, np.nan)
ppsf_q = df.groupby('crime_quintile', observed=True)['ppsf'].median()
bars2 = axes[1].bar(ppsf_q.index, ppsf_q.values, color=colors)
for bar, val in zip(bars2, ppsf_q.values):
    axes[1].text(bar.get_x()+bar.get_width()/2, val*1.01,
                 f'${val:.0f}/sqft', ha='center', fontsize=9, fontweight='bold')
axes[1].set_title('Median Price per SqFt by Crime Quintile\n(size-adjusted)', fontsize=11)
axes[1].set_ylabel('Median $/SqFt')

discount = (q_stats.loc['Q5\nHighest\ncrime','median_price'] /
            q_stats.loc['Q1\nLowest\ncrime','median_price'] - 1) * 100
plt.suptitle(f'Crime Discount: Highest vs. Lowest Crime Quintile: {discount:+.1f}%',
             fontsize=12, y=1.02)
plt.tight_layout()
plt.show()

print(q_stats.to_string())

In [ ]:
# ── Crime discount map: median price per sqft by neighborhood ─────────────────
nbhd_stats = df.groupby('neighborhood').agg(
    median_ppsf  = ('ppsf',              'median'),
    median_crime = ('crime_count_500m',  'median'),
    n_sales      = ('SalePrice',         'count'),
).query('n_sales >= 20').sort_values('median_crime')

fig, ax = plt.subplots(figsize=(12, 10))
sc = ax.scatter(
    nbhd_stats['median_crime'],
    nbhd_stats['median_ppsf'],
    s=nbhd_stats['n_sales'] / nbhd_stats['n_sales'].max() * 400,
    c=nbhd_stats['median_crime'], cmap='RdYlGn_r',
    alpha=0.8, edgecolors='white', lw=0.8
)

z = np.polyfit(nbhd_stats['median_crime'], nbhd_stats['median_ppsf'], 1)
xs = np.linspace(nbhd_stats['median_crime'].min(), nbhd_stats['median_crime'].max(), 100)
ax.plot(xs, np.poly1d(z)(xs), 'steelblue', ls='--', lw=1.5, alpha=0.7)

for nbhd, row in nbhd_stats.iterrows():
    ax.annotate(str(nbhd).title(), (row['median_crime'], row['median_ppsf']),
                fontsize=7, textcoords='offset points', xytext=(4,2), alpha=0.8)

plt.colorbar(sc, ax=ax, label='Median serious crimes within 500m (12-month window)')
ax.set_xlabel('Median Crime Exposure Score (500m, 12-month)', fontsize=11)
ax.set_ylabel('Median Price per SqFt ($)', fontsize=11)
ax.set_title('Neighborhood Crime Exposure vs. Price per SqFt\n(bubble size = transaction volume)',
             fontsize=12)
r_nbhd = nbhd_stats['median_crime'].corr(nbhd_stats['median_ppsf'])
ax.text(0.97, 0.97, f'r = {r_nbhd:.2f}', transform=ax.transAxes,
        ha='right', va='top', fontsize=11, color='steelblue')
plt.tight_layout()
plt.show()

**Conclusions:**
- *(To be filled after execution)*
- *(Key questions: What is the crime discount Q5 vs Q1? Is it larger for violent crime? Which Seattle neighborhoods show the strongest crime-price relationship?)*

---
## 6. Summary

| Status | Item |
|--------|------|
| ✅ Done | SPD crime data 2015–2024 downloaded (~700K records with valid coordinates) |
| ✅ Done | Crime trends, seasonality, and type breakdown |
| ✅ Done | Neighborhood crime heat map (`crime_data/map_crime_heatmap.html`) |
| ✅ Done | Spatio-temporal decay feature: crimes within 500m / 12 months per sale |
| ✅ Done | Crime quintile discount analysis (Seattle SFR 2016–2024) |
| ⏳ Pending | Fill in conclusion cells after reviewing charts |
| ⏳ Optional | Combine crime score + school quality score into a unified liveability index |
| ⏳ Optional | Extend to all of King County if broader crime data becomes available |